# 05 — Valutazione e calibrazione
Un decision model restituisce probabilità, quindi misuriamo sia **accuratezza** sia **calibrazione**:
- Brier score ed ECE (più bassi sono meglio);
- diagramma di affidabilità;
- soglia di confidenza che passa le decisioni incerte a un modello più forte.

Riferimenti gratuiti: [JevBench](https://github.com/fstandhartinger/jevbench) (MIT), [ProbabilityBench](https://github.com/ishida-lab/ProbabilityBench), [Evaluating & Benchmarking Jev](https://arxiv.org/abs/2609.37647), [Early Evidence Audit](https://arxiv.org/abs/2609.32160) (checklist a 14 punti), [Beyond Calibration](https://arxiv.org/abs/2609.33209).


In [ ]:
from decision_qa import MockClient, read_jsonl, brier, ece, fit_threshold
from decision_qa.evaluate import gold_probability

records = read_jsonl('data/pubmedqa_decisions.jsonl')
client = MockClient(seed=42, quality=0.75)
gold_probs, correct = [], []
for r in records:
    for q, res in zip(r.questions, client.decision(r.state, r.questions)):
        p = gold_probability(res, q)
        gold_probs.append(p)
        correct.append(int(p >= max(res.probabilities.values()) - 1e-12))
print('n =', len(gold_probs), '| accuracy =', round(sum(correct)/len(correct), 3),
      '| Brier =', round(brier(gold_probs), 3), '| ECE =', round(ece(gold_probs, correct), 3))


In [ ]:
from decision_qa.evaluate import reliability
rows = reliability(gold_probs, correct)
for row in rows:
    print(row['bin'], 'n=', row['count'], 'conf=', round(row['confidence'], 2),
          'acc=', round(row['accuracy'], 2))


In [ ]:
threshold = fit_threshold(gold_probs, correct, target_precision=0.9)
coverage = sum(p >= threshold for p in gold_probs) / len(gold_probs)
print(f'threshold={threshold:.2f} -> {coverage:.1%} of decisions can be committed')


Lettura critica: [JevAdvBench](https://arxiv.org/abs/2609.31142) mostra che lo stato è input non attendibile; [Jev Does Not Play Dice](https://kantahayashiai.github.io/posts/jev-does-not-play-dice/) mostra che le probabilità riportate possono essere male calibrate.
